# TrainLM: start pretraining from raw text

Open this notebook in Colab or Kaggle with a **single-device native TorchTPU runtime**. TrainLM builds a randomly initialized GPT-2 model, learns a tokenizer from a pinned Hugging Face dataset, prepares packed data, trains, evaluates, and exports a normal Hugging Face model.

For a first run, choose the token budget and sequence length. The default corpus is a pinned TinyStories revision. This is an experimental, small-scale workflow; it does not establish convergence or TPU performance.

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-scratch-source-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "main",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
from importlib.util import find_spec
missing = [
    requirement
    for module, requirement in (("datasets", "datasets>=3,<5"), ("tokenizers", "tokenizers>=0.21,<1"))
    if find_spec(module) is None
]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")

## 1. Choose your run

Edit the token budget and sequence length below. You can also choose another Hugging Face dataset by setting its owner/name, immutable 40-character revision, separate train/validation splits, and text column. The setup cell installs only the raw-text `datasets` and `tokenizers` helpers if missing; it leaves the preinstalled Torch/TorchTPU stack alone. TrainLM handles tokenization, data packing, model setup, runtime checks, and training arguments.

In [ ]:
from pathlib import Path
import sys
import tempfile

sys.path.insert(0, str(repo_root / "src"))
from trainlm import TrainLMPretrainingConfig, TrainLMTrainer

# Beginner settings
TARGET_SUPERVISED_TOKENS = 10_000
SEQUENCE_LENGTH = 128

# Optional data settings; keep the pinned defaults for your first run.
DATASET_ID = "roneneldan/TinyStories"
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
TRAIN_SPLIT = "train"
VALIDATION_SPLIT = "validation"
TEXT_FIELD = "text"

session_dir = Path(tempfile.mkdtemp(prefix="trainlm-pretraining-"))
output_dir = session_dir / "artifacts"
report_path = session_dir / "report.json"
pretraining_config = TrainLMPretrainingConfig(
    output_dir=output_dir,
    dataset_id=DATASET_ID,
    dataset_revision=DATASET_REVISION,
    train_split=TRAIN_SPLIT,
    validation_split=VALIDATION_SPLIT,
    text_field=TEXT_FIELD,
    max_tokens=TARGET_SUPERVISED_TOKENS,
    sequence_length=SEQUENCE_LENGTH,
    accelerator="native_tpu",
)
print(f"source_revision={source_revision}")
print(f"target_supervised_tokens={TARGET_SUPERVISED_TOKENS}")
print(f"output_dir={output_dir}")

## 2. Train and export

The public `TrainLMTrainer.from_scratch()` API prepares the tokenizer, packed data, and starter model. Then use the same familiar `train()`, `evaluate()`, and `save_model()` methods as other TrainLM workflows. Native TorchTPU remains experimental; restart the notebook runtime if initialization fails and leaves the TPU runtime unavailable.

In [ ]:
import json
import math

try:
    trainer = TrainLMTrainer.from_scratch(pretraining_config)
    state = trainer.train()
    evaluation = trainer.evaluate()
    export_dir = trainer.save_model(output_dir / "baseline-export")
    train_loss = float(state.loss) if state.loss is not None else None
    eval_loss = float(evaluation["eval_loss"])
    status = (
        "passed"
        if state.tokens_seen >= TARGET_SUPERVISED_TOKENS
        and train_loss is not None and math.isfinite(train_loss)
        and math.isfinite(eval_loss)
        else "failed"
    )
    report = {
        "schema_version": 1,
        "scope": "experimental_single_device_raw_text_pretraining",
        "status": status,
        "source_revision": source_revision,
        "dataset_id": DATASET_ID,
        "dataset_revision": DATASET_REVISION,
        "device": "native_tpu",
        "training": {
            "step": state.step,
            "supervised_tokens": state.tokens_seen,
            "target_supervised_tokens": TARGET_SUPERVISED_TOKENS,
            "train_loss": train_loss,
            "eval_loss": eval_loss,
        },
        "model_export": str(export_dir),
        "limitations": ["experimental_single_device", "does_not_certify_performance"],
    }
except Exception as exc:
    report = {
        "schema_version": 1,
        "scope": "experimental_single_device_raw_text_pretraining",
        "status": "failed",
        "source_revision": source_revision,
        "error_type": type(exc).__name__,
        "limitations": ["experimental_single_device", "does_not_certify_performance"],
    }
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print(json.dumps(report, indent=2))
    raise RuntimeError(f"Pretraining failed; download the report at {report_path}.") from None

report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
print(json.dumps(report, indent=2))
print(f"shareable_report={report_path}")
if report["status"] != "passed":
    raise RuntimeError(f"Training did not meet its token/loss checks; see {report_path}.")

## 3. Download the result

The ZIP contains the Hugging Face model/tokenizer export and run report. Colab starts a download; Kaggle saves the ZIP under `/kaggle/working` for the notebook output panel. If preflight or training fails, run the next cell to download the report without a model export.

In [ ]:
from pathlib import Path
import shutil
import zipfile

export_dir = session_dir / "artifacts" / "baseline-export"
required = ("config.json", "tokenizer_config.json")
missing = [name for name in required if not (export_dir / name).is_file()]
weights_present = (
    any(path.is_file() and path.suffix in {".safetensors", ".bin"}
        for path in export_dir.rglob("*"))
    if export_dir.is_dir() else False
)
model_export_ready = not missing and weights_present
if not report_path.is_file():
    raise RuntimeError("The evidence report is missing.")

artifact_bundle = session_dir / "trainlm-scratch-artifact.zip"
with zipfile.ZipFile(artifact_bundle, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(report_path, arcname="report.json")
    if model_export_ready:
        for path in sorted(export_dir.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=Path("model") / path.relative_to(export_dir))
if model_export_ready:
    print("model_export=included")
else:
    print(f"model_export=not_included missing_files={missing} weights_present={weights_present}")

kaggle_output = Path("/kaggle/working")
if kaggle_output.is_dir():
    kaggle_artifact = kaggle_output / artifact_bundle.name
    shutil.copy2(artifact_bundle, kaggle_artifact)
    print(f"kaggle_artifact={kaggle_artifact}")
try:
    from google.colab import files
except ImportError:
    print(f"artifact_bundle={artifact_bundle}")
else:
    files.download(str(artifact_bundle))